# 02a. Data Processing: Batch Data Cleaning on SerpApi Response

This notebook executes the ingestion and transformation pipeline for raw SerpApi JSON responses. It ensures that unstructured data is standardised, cleaned, and archived, transforming raw API outputs into a model-ready dataset.

## Objectives 
- Data Transformation: Parse, flatten, and compile new JSON response files into serpapi_response dataset.
- Feature Engineering: Derive critical analytical features:
  - Time-based Features: days_to_departure, day_of_week, day_name, is_weekend, and booking_window.
  - Label Features: other_airport and airline_code
- Data Governance & Archival: Maintain pipeline hygiene by systematically moving processed JSON files and legacy response versions into designated archival directories.

### Environment Setup

In [1]:
import json
import os
import shutil
import pandas as pd
from datetime import datetime

In [2]:
# Path setup
source_dir = './'
master_csv = 'serpapi_response.csv'
archive_dir = './serpapi_response/serpapi_response_archive'
processed_json_dir = './serpapi_response/Processed serpapi_response'

# Ensure target directories exist dynamically
os.makedirs(archive_dir, exist_ok = True)
os.makedirs(processed_json_dir, exist_ok = True)

### Data Processing and Archival

In [3]:
# Create list of strings like "serpapi_response"
json_files = [
    os.path.join(source_dir, file)
    for file in os.listdir(source_dir) 
    if file.endswith(".json")
]

print(f"Found {len(json_files)} files to process")

Found 4 files to process


In [4]:
# Archive current serpapi response
if os.path.exists(master_csv):

    # Archiving serpapi response and adding today's date
    today_str = datetime.now().strftime('%Y-%m-%d')
    archive_file_name = f"serpapi_response_{today_str}.csv"
    archive_path = os.path.join(archive_dir, archive_file_name)

    # Move the current file into the archive folder
    shutil.move(master_csv, archive_path)
    print(f"Sucessfully archived old master file to: {archive_path}")

Sucessfully archived old master file to: ./serpapi_response/serpapi_response_archive\serpapi_response_2026-05-29.csv


In [5]:
# Initialise master storage container
all_flattened_data = []

In [6]:
# Loop through filenames to process
for file in json_files: 
    # Track sucess
    processing_successful = False
    
    try: 
        with open(file, "r", encoding="utf-8") as f:
            batch_data = json.load(f)
    
        # Iterate through route keys (i.e., SIN-BKK, BKK_SIN)     
        for route_label, data in batch_data.items():
            if not isinstance(data,dict):
                print(f"Skipping {route_label}: Data is not a dictionary (it's a {type(data)}).")
                continue

            if "search_parameters" not in data:
                print(f"Skipping {route_label}: Missing 'search_parameters'.")
                continue
        
            search_metadata = data.get("search_metadata", {})
            search_parameters = data.get("search_parameters", {})
    
            date_str = search_metadata.get("processed_at", "")[:10]
            departure_str = search_parameters.get("outbound_date", "")
    
            # Parse date 
            date = datetime.strptime(date_str, "%Y-%m-%d")
            departure_date = datetime.strptime(departure_str, "%Y-%m-%d")

            # Creating time-based features
            days_to_departure = (departure_date - date).days
            day_of_week = departure_date.weekday() 
            day_name = departure_date.strftime("%A")
            is_weekend = True if day_of_week in [4,5,6] else False
    
            # Booking Window logic
            if days_to_departure <= 0:
                booking_window = "0 days"
            else: 
                lower_bound = ((days_to_departure -1) // 14)* 14 + 1
                upper_bound = lower_bound + 13
                booking_window = f"{lower_bound}-{upper_bound} days"

            # Creating label features
            departure_airport = search_parameters.get("departure_id")
            arrival_airport = search_parameters.get("arrival_id")
            other_airport = arrival_airport if departure_airport == "SIN" else departure_airport
    
            all_itineraries = data.get("best_flights",[]) + data.get("other_flights",[])
    
            for flight in all_itineraries: 
                price = flight.get("price")
                if not price:
                    continue
    
                flights_segments = flight.get("flights", [])
                first_segment = flights_segments[0] if flights_segments else {}
                
                airline = first_segment.get("airline", "Unknown Airline")
                raw_flight_num = first_segment.get("flight_number", "")
                airline_code = raw_flight_num[:2] if raw_flight_num else "NIL"
            
                row_record = {
                    "date": date_str,
                    "route": f"{departure_airport}-{arrival_airport}",
                    "departure_date": departure_str,
                    "airline": airline,
                    "airline_code": airline_code,
                    "price": float(price),
                    "days_to_departure": days_to_departure,
                    "day_of_week": day_of_week,
                    "day_name": day_name,
                    "is_weekend": is_weekend,
                    "departure_airport":departure_airport,
                    "out_inbound": 1 if departure_airport == "SIN" else 2,
                    "other_airport": other_airport,
                    "data_source": "api",
                    "booking_window": booking_window
                }
    
                # Append each row to the master list
                all_flattened_data.append(row_record)
            
            # Mark as successfull once loop completes without error
            processing_successful = True

    except Exception as e:
        print(f"Error processing file {file}: {e}")

    # Archive processed JSON files
    if processing_successful: 
        file_name = os.path.basename(file)
        dest_path = os.path.join(processed_json_dir, file_name)
        shutil.move(file, dest_path)
        print(f"Moved processed file to: {dest_path}")
    else: 
        print(f"{file} left in source directory due to processing failure.")


Moved processed file to: ./serpapi_response/Processed serpapi_response\serpapi_response_2026-05-30_search2026-05-29.json
Moved processed file to: ./serpapi_response/Processed serpapi_response\serpapi_response_2026-06-11_search2026-05-29.json
Moved processed file to: ./serpapi_response/Processed serpapi_response\serpapi_response_2026-07-15_search2026-05-29.json
Moved processed file to: ./serpapi_response/Processed serpapi_response\serpapi_response_2026-08-03_search2026-05-29.json


### Data Export

In [7]:
# Save master CSV
if all_flattened_data: 
    final_df = pd.DataFrame(all_flattened_data)
    final_df.to_csv(master_csv,index=False)
    print(f"Serpapi cleaning complete. New batch saved as: {master_csv}")
else:
    print(f"No new data found to parse this execution run.")

Serpapi cleaning complete. New batch saved as: serpapi_response.csv
